# 🏘 DT-HRES-S — Simulador Comunitario
### 2D Body: la interfaz para tomadores de decisiones

Este notebook es el primer prototipo de la interfaz que verá la comunidad de Ixil para diseñar su sistema híbrido de energía renovable.

**Cómo usarlo:** mueve los sliders, marca las casillas, y presiona el botón **Calcular**. El digital twin estimará en segundos cuál es el sistema óptimo.

---
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Aaron-Cuevas/DT-HRES-S/blob/main/notebooks/12_community_interface.ipynb)

## Setup

In [ ]:
import os, sys

if 'google.colab' in sys.modules and not os.path.exists('DT-HRES-S'):
    !git clone -q https://github.com/Aaron-Cuevas/DT-HRES-S.git
    %cd DT-HRES-S
    !pip install -q -r requirements.txt
elif os.path.basename(os.getcwd()) == 'notebooks':
    sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

from src.data_loader import load_city, CITIES
from src import pv_model, wind_model, battery_model
from src import hres_simulator as hres
from src.uncertainty import full_budget_for_pv

print('✓ Setup complete — ready to build the interface')

## 🏗 Construir la interfaz

Los sliders representan **decisiones de la comunidad** (cuántas casas, qué servicios, qué presupuesto).

In [ ]:
# === Widgets de entrada ===

# Comunidad
ciudad_proxy = widgets.Dropdown(
    options=['Campeche (proxy de Ixil)', 'Monterrey', 'Mexico City', 'San Ignacio'],
    value='Campeche (proxy de Ixil)',
    description='Sitio:',
)
casas = widgets.IntSlider(value=20, min=5, max=100, step=1, description='Casas:',
                          continuous_update=False)

# Servicios comunitarios
iluminacion = widgets.Checkbox(value=True, description='Iluminación básica (LED)')
refrigeracion = widgets.Checkbox(value=True, description='Refrigeración doméstica')
escuela = widgets.Checkbox(value=True, description='Escuela (5 computadoras)')
clinica = widgets.Checkbox(value=False, description='Clínica (vacunas refrigeradas)')
bomba_agua = widgets.Checkbox(value=False, description='Bomba de agua')

# Tamaño del sistema
kwp_pv = widgets.FloatSlider(value=4.0, min=1.0, max=15.0, step=0.5,
                              description='PV (kWp):', continuous_update=False)
kw_wind = widgets.FloatSlider(value=3.0, min=0.0, max=10.0, step=0.5,
                               description='Eólico (kW):', continuous_update=False)
kwh_bat = widgets.FloatSlider(value=10.0, min=2.0, max=40.0, step=1.0,
                               description='Batería (kWh):', continuous_update=False)

# Display
section_comunidad = widgets.VBox([
    widgets.HTML('<h3>🏘 Tu comunidad</h3>'),
    ciudad_proxy, casas,
    widgets.HTML('<h4>Servicios eléctricos:</h4>'),
    iluminacion, refrigeracion, escuela, clinica, bomba_agua,
])

section_sistema = widgets.VBox([
    widgets.HTML('<h3>⚡ Tu sistema híbrido propuesto</h3>'),
    kwp_pv, kw_wind, kwh_bat,
])

boton_calcular = widgets.Button(
    description='🔄 Calcular',
    button_style='success',
    layout=widgets.Layout(width='200px', height='40px'),
)

output = widgets.Output()

display(section_comunidad, section_sistema, boton_calcular, output)

## 🎯 Función de cálculo

Cuando el usuario presione **Calcular**, esta función traduce sus elecciones en una simulación completa.

In [ ]:
# Mapping del dropdown al nombre de ciudad real
CIUDAD_MAP = {
    'Campeche (proxy de Ixil)': 'Campeche',
    'Monterrey': 'Monterrey',
    'Mexico City': 'Mexico City',
    'San Ignacio': 'San Ignacio',
}

# Demanda estimada por servicio (kWh/día por hogar para servicios domésticos,
# o kWh/día absolutos para servicios comunitarios)
DEMANDA_SERVICIOS = {
    'iluminacion':    0.5,    # kWh/casa/día con LEDs
    'refrigeracion':  1.2,    # kWh/casa/día (refri eficiente)
    'escuela':       10.0,    # kWh/día absoluto
    'clinica':       15.0,    # kWh/día absoluto (refrigeración 24/7)
    'bomba_agua':    20.0,    # kWh/día absoluto
}

def estimar_demanda_diaria(n_casas, servicios):
    """kWh/día totales para la comunidad."""
    e_por_casa = 0
    if servicios['iluminacion']:   e_por_casa += DEMANDA_SERVICIOS['iluminacion']
    if servicios['refrigeracion']: e_por_casa += DEMANDA_SERVICIOS['refrigeracion']
    e_total = e_por_casa * n_casas
    if servicios['escuela']:       e_total += DEMANDA_SERVICIOS['escuela']
    if servicios['clinica']:       e_total += DEMANDA_SERVICIOS['clinica']
    if servicios['bomba_agua']:    e_total += DEMANDA_SERVICIOS['bomba_agua']
    return e_total

def estimar_costo(kwp_pv, kw_wind, kwh_bat):
    """Costo en MXN. Precios típicos del mercado mexicano 2025-2026."""
    cost_pv_per_kwp = 22000      # MXN/kWp panel + montaje
    cost_wind_per_kw = 35000     # MXN/kW turbina pequeña
    cost_bat_per_kwh = 12000     # MXN/kWh LiFePO4
    cost_balance = 30000         # inversor + controladores
    return (kwp_pv * cost_pv_per_kwp
            + kw_wind * cost_wind_per_kw
            + kwh_bat * cost_bat_per_kwh
            + cost_balance)

In [ ]:
def calcular(_=None):
    with output:
        clear_output(wait=True)

        ciudad = CIUDAD_MAP[ciudad_proxy.value]
        servicios = {
            'iluminacion': iluminacion.value,
            'refrigeracion': refrigeracion.value,
            'escuela': escuela.value,
            'clinica': clinica.value,
            'bomba_agua': bomba_agua.value,
        }
        e_diaria_kWh = estimar_demanda_diaria(casas.value, servicios)
        peak_kW = e_diaria_kWh / 12  # estimación gruesa

        # Construir el sistema
        n_panels = int(kwp_pv.value * 1000 / 400)  # paneles de 400 W
        config = hres.HRESConfig(
            pv=pv_model.PVSystem(p_rated_W=400, n_panels=n_panels, tilt_deg=20),
            wind=wind_model.WindTurbine(rated_power_W=int(kw_wind.value * 1000)) if kw_wind.value > 0 else None,
            battery=battery_model.Battery(capacity_kWh=kwh_bat.value),
            has_wind=(kw_wind.value > 0),
        )

        # Simular
        df = load_city(ciudad)
        demand_W = hres.synthetic_community_load(df, peak_kW=peak_kW,
                                                  base_kW=peak_kW * 0.1)
        sim = hres.run(df, config, demand_W=demand_W)
        kpis = hres.summarize(sim, config)

        costo = estimar_costo(kwp_pv.value, kw_wind.value, kwh_bat.value)
        unc = full_budget_for_pv('TMY_file')

        # Resultados — formato comunitario (Persona 1)
        emoji_cob = '🟢' if kpis['coverage_pct'] > 90 else ('🟡' if kpis['coverage_pct'] > 75 else '🔴')
        print('=' * 60)
        print('  RESULTADOS PARA TU COMUNIDAD')
        print('=' * 60)
        print()
        print(f'🏘 Casas: {casas.value} | Ubicación: {ciudad_proxy.value}')
        print(f'⚡ Demanda estimada: {e_diaria_kWh:.1f} kWh/día '
              f'({e_diaria_kWh * 365:.0f} kWh/año)')
        print()
        print('Tu sistema:')
        print(f'  ☀ PV:      {kpis["pv_kWp"]:.1f} kWp ({n_panels} paneles)')
        print(f'  🌪 Eólico:  {kpis["wind_kW"]:.1f} kW')
        print(f'  🔋 Batería: {kpis["battery_kWh"]:.1f} kWh')
        print()
        print('Desempeño esperado:')
        print(f'  {emoji_cob} Cobertura: {kpis["coverage_pct"]:.1f}% del tiempo tendrás electricidad')
        print(f'  📈 Energía renovable: {kpis["renewable_fraction"]*100:.0f}%')
        print(f'  ⚠ LPSP (energía no servida): {kpis["LPSP"]*100:.1f}%')
        print(f'  ⏱ Horas sin servicio al año: {kpis["LOLE_h"]:.0f}')
        print()
        print(f'💰 Inversión inicial estimada: ${costo:,.0f} MXN')
        print(f'   Por casa: ${costo / casas.value:,.0f} MXN')
        print()
        print(f'❓ Incertidumbre del modelo: ±{unc.combined_pct:.1f}% (95% CI)')
        print('   Esta estimación incluye errores de datos sintéticos. Con datos reales de Ixil, se reduce.')

        # Plot: una semana típica
        fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
        week = sim.iloc[24*120:24*127]  # finales de abril (verano)
        axes[0].plot(week.index, week['p_pv_W']/1000, label='PV', color='gold', linewidth=1.5)
        if kw_wind.value > 0:
            axes[0].plot(week.index, week['p_wind_W']/1000, label='Wind', color='steelblue', linewidth=1.5)
        axes[0].plot(week.index, week['p_demand_W']/1000, label='Demanda', color='black', linestyle='--', alpha=0.7)
        axes[0].set_ylabel('Potencia (kW)')
        axes[0].set_title('Una semana típica — abril/mayo')
        axes[0].legend()
        axes[0].grid(alpha=0.3)

        axes[1].fill_between(week.index, 0, week['soc']*100, alpha=0.5, color='green')
        axes[1].set_ylabel('Batería SoC (%)')
        axes[1].set_ylim(0, 100)
        axes[1].axhline(20, linestyle=':', color='red', alpha=0.5, label='Mínimo (20%)')
        axes[1].set_xlabel('Hora del año')
        axes[1].legend()
        axes[1].grid(alpha=0.3)
        plt.tight_layout()
        plt.show()

boton_calcular.on_click(calcular)

# Calcular automáticamente con valores por defecto
calcular()

## 🎓 Cómo interpretar los resultados

**Cobertura** — porcentaje del tiempo que tendrás electricidad.
- 🟢 >90%: aceptable
- 🟡 75-90%: aceptable solo con generador de respaldo
- 🔴 <75%: necesitas más componentes

**LPSP** (Loss of Power Supply Probability) — la fracción de la energía que el sistema NO logra entregar. Si es 5%, significa que el 5% de la energía demandada no fue suministrada.

**Fracción renovable** — qué porcentaje de tu electricidad vino del sol y viento (lo demás, en este modelo simplificado, sería déficit; en sistemas reales, sería diésel).

**Incertidumbre ±X%** — recuerda que esta simulación usa datos sintéticos (TMY de Campeche como proxy de Ixil). Cuando llegue la data real, esta incertidumbre se reducirá.

---

## 📋 Siguientes pasos

Esta interfaz es la **v0.2.0**. Roadmap:

- **v0.3:** integrar NASA POWER para que la opción 'Ixil' sea data real, no proxy
- **v0.4:** agregar optimizador automático ('encuentra el sistema más barato con cobertura > 90%')
- **v0.5:** versión en lengua maya yucateca
- **v1.0:** validación con datos medidos en el sitio real